# Coding Attention Mechanisms

### Different parts of the input with self-attention

A simple self-attention mechanism without trainable weights

In [ ]:
import torch
inputs=torch.tensor(
    [[0.43,0.15,0.89],  #Your
     [0.55,0.87,0.66],  #journey
     [0.57,0.85,0.64],  #starts
     [0.22,0.58,0.33],  #with
     [0.77,0.25,0.10],  #one
     [0.05,0.80,0.55]]  #step
)

In [ ]:
query=inputs[1]

attention_score1=torch.empty(inputs.shape[0])
for i,x_i in enumerate(inputs):
  attention_score1[i]=torch.dot(x_i,query)
print(attention_score1)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [ ]:
def softmax_naive(x):
  return torch.exp(x)/torch.exp(x).sum(dim=0)

softmax_naive(attention_score1)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [ ]:
query=inputs[1]
context_vec1=torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
  context_vec1+=attention_score1[i]*x_i
print(context_vec1)

tensor([2.8579, 4.2330, 3.7270])


### A simple self- attention mechanism without trainable weights

In [ ]:
attention_scores=torch.empty(6,6)
for i,x_i in enumerate(inputs):
  for j,x_j in enumerate(inputs):
    attention_scores[i,j]=torch.dot(x_i,x_j)

print(attention_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [ ]:
#using in-built matrix multiplication
attention_scores=inputs@inputs.T
attn_weights=torch.softmax(attention_scores,dim=1)
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [ ]:
all_context_vecs=attn_weights @ inputs
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

### Implementing self-attention with trainable weights

Computing attention weights step by step

In [ ]:
d_in=inputs.shape[1]
d_out=2

In [ ]:
torch.manual_seed(123)

W_query=torch.nn.Parameter(torch.rand(d_in,d_out))
W_key=torch.nn.Parameter(torch.rand(d_in,d_out))
W_value=torch.nn.Parameter(torch.rand(d_in,d_out))

In [ ]:
queries=inputs@W_query
keys=inputs@W_key
values=inputs@W_value

In [ ]:
attn_scores=queries@keys.T
d_k=keys.shape[1]
attn_weights=torch.softmax(attn_scores/ (d_k**0.5),dim=-1);
context_vecs=attn_weights@values
context_vecs

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

## Implementing a compact self-attention class

In [ ]:
import torch.nn as nn
class SelfAttention_v1(nn.Module):

  def __init__(self,d_in,d_out):
    super().__init__()
    self.W_query=nn.Parameter(torch.rand(d_in,d_out))
    self.W_key=nn.Parameter(torch.rand(d_in,d_out))
    self.W_value=nn.Parameter(torch.rand(d_in,d_out))

  def forward(self,x):
    queries=inputs@W_query
    keys=inputs@W_key
    values=inputs@W_value

    attn_scores=queries@keys.T
    d_k=keys.shape[0]
    attn_weights=torch.softmax(attn_scores/(d_k**0.5),dim=-1)
    context_vecs=attn_weights@values
    return context_vecs

torch.manual_seed(123)
sa_v1=SelfAttention_v1(d_in,d_out)
sa_v1(inputs)

tensor([[0.2915, 0.7857],
        [0.2955, 0.7957],
        [0.2953, 0.7952],
        [0.2886, 0.7786],
        [0.2874, 0.7757],
        [0.2912, 0.7849]], grad_fn=<MmBackward0>)

In [ ]:
class SelfAttention_v2(nn.Module):

  def __init__(self,d_in,d_out,qkv_bias=False):
    super().__init__()
    self.W_query=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_key=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_value=torch.nn.Linear(d_in,d_out,bias=qkv_bias)

  def forward(self,x):
    queries=self.W_query(x)
    keys=self.W_key(x)
    values=self.W_value(x)

    attn_scores=queries@keys.T
    d_k=keys.shape[0]
    attn_weights=torch.softmax(attn_scores/(d_k**0.5),dim=-1)
    context_vecs=attn_weights@values
    return context_vecs

torch.manual_seed(789)
sa_v2=SelfAttention_v2(d_in,d_out)
sa_v2(inputs)

tensor([[-0.0752,  0.0693],
        [-0.0757,  0.0688],
        [-0.0758,  0.0687],
        [-0.0765,  0.0676],
        [-0.0767,  0.0673],
        [-0.0761,  0.0682]], grad_fn=<MmBackward0>)

## Hiding future words with causal attention

Applying Causal Attention

In [ ]:
queries=sa_v2.W_query(inputs)
keys=sa_v2.W_key(inputs)
values=sa_v2.W_value(inputs)

attn_scores=queries@keys.T
d_k=keys.shape[0]
attn_weights=torch.softmax(attn_scores/d_k**0.5,dim=-1)
context_vecs=attn_weights@values

In [ ]:
context_length=attention_scores.shape[0]
mask_simple=torch.tril(torch.ones(context_length,context_length))
mask_simple

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [ ]:
masked_simple=mask_simple*attn_weights
masked_simple

tensor([[0.1811, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1876, 0.1665, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1873, 0.1665, 0.1667, 0.0000, 0.0000, 0.0000],
        [0.1781, 0.1668, 0.1668, 0.1611, 0.0000, 0.0000],
        [0.1760, 0.1668, 0.1669, 0.1621, 0.1662, 0.0000],
        [0.1818, 0.1666, 0.1667, 0.1595, 0.1667, 0.1587]],
       grad_fn=<MulBackward0>)

In [ ]:
row_sums=masked_simple.sum(dim=-1,keepdim=True)
masked_simple_norm=masked_simple/row_sums
masked_simple_norm

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5299, 0.4701, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3599, 0.3199, 0.3202, 0.0000, 0.0000, 0.0000],
        [0.2647, 0.2478, 0.2479, 0.2395, 0.0000, 0.0000],
        [0.2100, 0.1991, 0.1991, 0.1935, 0.1983, 0.0000],
        [0.1818, 0.1666, 0.1667, 0.1595, 0.1667, 0.1587]],
       grad_fn=<DivBackward0>)

In [ ]:
mask=torch.triu(torch.ones(context_length,context_length),diagonal=1)
masked=attn_scores.masked_fill(mask.bool(),-torch.inf)

In [ ]:
attn_weights=torch.softmax(masked,dim=-1)
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5728, 0.4272, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4000, 0.2996, 0.3004, 0.0000, 0.0000, 0.0000],
        [0.2870, 0.2441, 0.2444, 0.2245, 0.0000, 0.0000],
        [0.2251, 0.1974, 0.1976, 0.1842, 0.1957, 0.0000],
        [0.2054, 0.1659, 0.1662, 0.1490, 0.1662, 0.1472]],
       grad_fn=<SoftmaxBackward0>)

Masking additional attention weights with droput

In [ ]:
torch.manual_seed(123)
dropout=torch.nn.Dropout(0.5)

In [ ]:
example=torch.ones(6,6)
example

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [ ]:
dropout(example)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

In [ ]:
dropout_rate=0.8
1/(1-dropout_rate)

5.000000000000001

In [ ]:
dropout=torch.nn.Dropout(dropout_rate)
dropout(example)

tensor([[5., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 5., 0., 5., 0.],
        [0., 0., 5., 0., 0., 0.],
        [0., 0., 0., 0., 5., 0.],
        [5., 5., 0., 0., 5., 5.]])

Implementing a compact causal self-attention class

In [ ]:
batch=torch.stack((inputs,inputs),dim=0)
batch

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])

In [ ]:
import torch
class Causal_Attention(torch.nn.Module):

  def __init__(self,d_in,d_out,context_length,dropout,qkv_bias=False):
    super().__init__()
    self.W_query=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_key=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_value=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
    self.dropout=torch.nn.Dropout(dropout)
    self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))

  def forward(self,x):
    b,num_tokens,d_in=x.shape
    queries=self.W_query(x)
    keys=self.W_key(x)
    values=self.W_value(x)

    attn_scores=queries@keys.transpose(1,2)
    attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens],-torch.inf)
    attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
    context_vecs=attn_weights@values
    return context_vecs

context_length=batch.shape[1]
dropout=0.0
torch.manual_seed(789)
ca=Causal_Attention(d_in,d_out,6,0.5,False)
ca(batch)

tensor([[[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]],

        [[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]]], grad_fn=<UnsafeViewBackward0>)

### Extending single-head attention to multi-head attention

Stacking multiple single-head attention layers

In [ ]:
class MultiHeadAttentionWrapper(torch.nn.Module):
  def __init__(self,d_in,d_out,context_length,dropout,num_heads=2,qkv_bias=False):
    super().__init__()
    self.heads=nn.ModuleList([
        Causal_Attention(d_in,d_out,context_length,dropout,qkv_bias) for _ in range(num_heads)
    ])

  def forward(self,x):
    return torch.cat([head(x) for head in self.heads],dim=-1)

torch.manual_seed(123)
context_length=batch.shape[1]
d_in,d_out=3,2
mha=MultiHeadAttentionWrapper(d_in,d_out,context_length,dropout=0.0,num_heads=2)
mha(batch)


tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)

### Multihead Attention with weight splits

In [ ]:
class MultiHeadAttention(torch.nn.Module):
  def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
    super().__init__()
    assert(d_out%num_heads==0), \
    "d_out must be divisible by the number of heads"

    self.d_out=d_out
    self.num_heads=num_heads
    self.head_dim=d_out//num_heads

    self.W_query=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_key=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_value=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
    self.out_proj=torch.nn.Linear(d_out,d_out)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))

  def forward(self,x):
    b,num_tokens,d_in=x.shape

    keys=self.W_key(x)
    queries=self.W_query(x)
    values=self.W_value(x)

    keys=keys.view(b,num_tokens,self.num_heads,self.head_dim)
    values=values.view(b,num_tokens,self.num_heads,self.head_dim)
    queries=queries.view(b,num_tokens,self.num_heads,self.head_dim)

    keys=keys.transpose(1,2)
    queries=queries.transpose(1,2)
    values=values.transpose(1,2)

    attn_scores=queries@keys.transpose(2,3)
    mask_bool=self.mask.bool()[:num_tokens,:num_tokens]
    attn_scores.masked_fill_(mask_bool,-torch.inf)

    attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
    attn_weights=self.dropout(attn_weights)

    context_vec=attn_weights@values
    context_vec=context_vec.contiguous().view(b,num_tokens,self.d_out)
    context_vec=self.out_proj(context_vec)

    return context_vec

torch.manual_seed(123)
batch_size,context_length,d_in=batch.shape
d_out=4
mha=MultiHeadAttention(d_in,d_out,context_length,0.0,num_heads=2)

context_vecs=mha(batch)
print(context_vecs)

tensor([[[ 0.3653,  0.3094, -0.0353, -0.7111],
         [ 0.2501,  0.2514, -0.0313, -0.5346],
         [ 0.2187,  0.2477, -0.0308, -0.4916],
         [-0.2922,  0.2364, -0.1030, -0.0846],
         [-0.3720,  0.1795, -0.0977,  0.0531],
         [-0.3209,  0.1911, -0.0927,  0.0013]],

        [[ 0.3653,  0.3094, -0.0353, -0.7111],
         [ 0.2501,  0.2514, -0.0313, -0.5346],
         [ 0.2187,  0.2477, -0.0308, -0.4916],
         [-0.2922,  0.2364, -0.1030, -0.0846],
         [-0.3720,  0.1795, -0.0977,  0.0531],
         [-0.3209,  0.1911, -0.0927,  0.0013]]], grad_fn=<ViewBackward0>)
